# GSM8K via distillation — choose the student (VishwamAI base or a ~1.5B open math model)

`STUDENT` selects what is trained. `'VishwamAI/vishwamai-base-v0'` is the from-scratch VishwamAI base (132.7M total / 54.9M active parameters, **full fine-tune**); it cannot realistically reach 50% on GSM8K, expect low single digits to roughly 10%. An **open ~1.5B math-strong model** (the default) is fine-tuned with LoRA on teacher solutions that are filtered for correct final answers, and measures GSM8K with greedy decoding.

**Read before running**
* Run it once per student and compare: each run appends its before/after scores to `WORK/math/results.json` and the last cell prints one table.
* The student is evaluated **before** training. If it already scores above 50%, that score comes from the student's own pre-training, not from this distillation. The notebook prints both numbers so you can tell.
* Check the licence of `STUDENT` and of every teacher before publishing, and say on the model card that the weights derive from them. Name the result accordingly (it is a derived model, not from-scratch VishwamAI).
* Runs on **Colab** (Drive for storage) or **Kaggle** (`/kaggle/working`; turn Internet ON in the notebook settings and add `HF_TOKEN` under Add-ons → Secrets). Free T4: expect a few thousand verified examples and 1-2 hours per stage, not the 100k+ that a strong result usually uses. GSM8K scores on 300 test questions carry about ±3 points of noise; set `N_EVAL = 1319` for the full test set.
* Training data uses GSM8K **train** only and is checked for overlap with the GSM8K test questions.

## 1 · Setup

In [ ]:
!pip -q install -U transformers peft datasets accelerate bitsandbytes

In [ ]:
import os, json, re, glob, random, math, shutil
import numpy as np, torch

# ---- where files live: Kaggle working dir, else Google Drive on Colab, else the current folder ----
if os.path.exists('/kaggle/working'):
    WORK = '/kaggle/working/vishwamai'
else:
    try:
        from google.colab import drive
        drive.mount('/content/drive'); WORK = '/content/drive/MyDrive/vishwamai'
    except Exception:
        WORK = './vishwamai'
print('work dir:', WORK)
os.makedirs(f'{WORK}/math', exist_ok=True)

STUDENT = 'VishwamAI/vishwamai-base-v0'    # or 'Qwen/Qwen2.5-Math-1.5B'
FULL_FT = STUDENT.startswith('VishwamAI/')   # tiny model: full fine-tune in fp32 instead of LoRA
SLUG = STUDENT.split('/')[-1]
MAX_LEN, N_EVAL, SEED = 768, 300, 0
MATH_SUFFIX = "\n\nEnd your reply with a final line '#### <number>'."
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
SKIP_TEACHER = True    # True: skip the hours-long teacher generation (section 4) and train on the human GSM8K solutions only

# ---------------- Hugging Face token ----------------
# Looks in: environment -> Kaggle Secrets / Colab Secrets (HF_TOKEN or hf_token) -> a saved login -> asks you to type one (interactive only).
HOW_TO_MAKE_TOKEN = """
No Hugging Face token found. To create one:
  1. Open https://huggingface.co/settings/tokens  ->  'Create new token'
  2. Type: 'Read' is enough (to publish later use 'Write'). If the VishwamAI repo is private, the token's account must have access to it.
  3. Copy the token (it starts with hf_...). Kaggle: Add-ons > Secrets > Add Secret, label HF_TOKEN, paste it, and TICK the checkbox so it is
     attached to this notebook. Colab: key icon > Add new secret, name HF_TOKEN, Notebook access on. (Secrets cannot be created from code.)
  4. Re-run this cell.  For a quick interactive test you can also just paste the token into the prompt that appears."""

def find_hf_token():
    t = os.environ.get('HF_TOKEN')
    if t: return t, 'environment'
    for label in ('HF_TOKEN', 'hf_token'):
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(label), f"Kaggle secret '{label}'"
        except Exception: pass
        try:
            from google.colab import userdata
            return userdata.get(label), f"Colab secret '{label}'"
        except Exception: pass
    try:
        from huggingface_hub import get_token
        t = get_token()
        if t: return t, 'saved Hugging Face login'
    except Exception: pass
    return None, None

HF_TOKEN, _src = find_hf_token()
if not HF_TOKEN and STUDENT.startswith('VishwamAI/'):            # the base repo is private: a token is required
    print(HOW_TO_MAKE_TOKEN)
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') == 'Batch':      # a saved run cannot ask
        raise RuntimeError('HF_TOKEN missing in a saved run: create the Kaggle secret as described above, then Save Version again.')
    import getpass
    HF_TOKEN = getpass.getpass('Paste your Hugging Face token (hidden): ').strip(); _src = 'typed in'
if HF_TOKEN:
    from huggingface_hub import HfApi
    try:
        print(f"Hugging Face token found ({_src}); logged in as:", HfApi().whoami(token=HF_TOKEN)['name'])
    except Exception as e:
        raise RuntimeError(f'The token was rejected ({type(e).__name__}). Create a new one as described above.')
    os.environ['HF_TOKEN'] = HF_TOKEN

print('GPU:', torch.cuda.get_device_name(0), '| fp16 only on T4 (no bf16)')

## 2 · GSM8K data and helpers

In [ ]:
from datasets import load_dataset
gtr = load_dataset('openai/gsm8k', 'main', split='train')
gte = load_dataset('openai/gsm8k', 'main', split='test')
def gold(a): return a.split('####')[-1].strip().replace(',', '')
def make_prompt(q): return f"Question: {q}\nSolve step by step and end with '#### <number>'.\nAnswer:"

def extract(text):
    m = re.findall(r'####\s*(-?[\d,]*\.?\d+)', text)
    if m: return m[0].replace(',', '')
    m = re.findall(r'-?\d[\d,]*\.?\d*', text)            # fallback: last number
    return m[-1].replace(',', '').rstrip('.') if m else None
def same(a, b):
    try: return abs(float(a) - float(b)) < 1e-4
    except Exception: return False

# decontamination: drop any training question sharing an 8-word window with a GSM8K *test* question
def grams(t, n=8):
    w = re.findall(r'\w+', t.lower()); return {' '.join(w[i:i + n]) for i in range(max(0, len(w) - n + 1))}
TEST_GRAMS = set().union(*[grams(q) for q in gte['question']])
def contaminated(q): return bool(grams(q) & TEST_GRAMS)
print('train', len(gtr), '| test', len(gte), '| test 8-grams', len(TEST_GRAMS))
print('contaminated train questions:', sum(contaminated(q) for q in gtr['question']), '(expected ~0)')

## 3 · Load the student and measure it BEFORE training

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
HF_TOKEN = os.environ.get('HF_TOKEN')                   # set by the setup cell (None for a public student)

dtype = torch.float32 if FULL_FT else torch.float16
tok = AutoTokenizer.from_pretrained(STUDENT, token=HF_TOKEN)
if tok.pad_token is None: tok.pad_token = tok.eos_token
try:                                                    # newer transformers: `dtype`; older: `torch_dtype`
    model = AutoModelForCausalLM.from_pretrained(STUDENT, dtype=dtype, token=HF_TOKEN)
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(STUDENT, torch_dtype=dtype, token=HF_TOKEN)
model = model.to('cuda:0')
NG = torch.cuda.device_count(); print('GPUs:', NG)

# VishwamAI was pre-trained with every sequence starting with the EOS token (its BOS convention)
BOS_IDS = [tok.eos_token_id] if FULL_FT else []
def encode_prompt(q): return BOS_IDS + tok(make_prompt(q), add_special_tokens=False)['input_ids']

import copy, concurrent.futures as cf

@torch.no_grad()
def evaluate(m, n=N_EVAL, bs=75, max_new=256):   # big batches: these small models leave the GPUs idle at bs=16
    """Greedy GSM8K accuracy. With 2 GPUs a copy of the model runs on each and they take alternate batches."""
    m.eval(); qs, ans = gte['question'][:n], [gold(a) for a in gte['answer'][:n]]
    reps = [m] + [copy.deepcopy(m).to(f'cuda:{g}').eval() for g in range(1, NG)]
    batches = [(qs[i:i + bs], ans[i:i + bs]) for i in range(0, len(qs), bs)]
    def work(rep, mine):
        dev = next(rep.parameters()).device; ok = 0
        for qb, ab in mine:
            ids = [encode_prompt(q) for q in qb]; L = max(map(len, ids))
            inp = torch.tensor([[tok.pad_token_id] * (L - len(x)) + x for x in ids]).to(dev)        # left padding
            att = torch.tensor([[0] * (L - len(x)) + [1] * len(x) for x in ids]).to(dev)
            out = rep.generate(input_ids=inp, attention_mask=att, max_new_tokens=max_new, do_sample=False,
                               pad_token_id=tok.pad_token_id)
            for o, a in zip(out[:, L:], ab):
                t = tok.decode(o, skip_special_tokens=True).split('\nQuestion:')[0]
                ok += same(extract(t), a)
        return ok
    with cf.ThreadPoolExecutor(len(reps)) as ex:
        oks = list(ex.map(work, reps, [batches[g::len(reps)] for g in range(len(reps))]))
    del reps
    torch.cuda.empty_cache()
    return sum(oks) / len(qs)

In [ ]:
base_acc = evaluate(model)
print(f'student BEFORE distillation: {100 * base_acc:.1f}% on {N_EVAL} GSM8K test questions (greedy)')

## 4 · Teacher data (open models, run locally)
Builds prompts from GSM8K **train** with the known answer, then runs the same generator as the VishwamAI notebook (`DeepSeek-R1-Distill-Qwen-7B`, 4-bit). Only solutions whose final number matches the known answer are kept. Re-running resumes where it stopped. Skip these cells if `voted.jsonl` / `out_*.jsonl` already exist in `WORK/distill`.

**Optional:** this stage takes hours on a free GPU. If you skip it, the SFT set below uses only the 7,473 human GSM8K solutions, which is fine for the small VishwamAI student.

This whole section is controlled by `SKIP_TEACHER` in the setup cell (default `True`: skipped). Set it to `False` to generate teacher solutions.

In [ ]:
%%writefile distill_generate.py
"""Sequence-level distillation data generator for VishwamAI (local weights, no API).

Loads an open-weight teacher from safetensors with transformers and writes JSONL
ready to merge into the Stage-3 SFT data of vishwamai_pretrain.ipynb.

Teachers (override the repo with --model):
    deepseek  deepseek-ai/DeepSeek-R1-Distill-Qwen-7B   (the full 671B R1 does not fit one GPU)
    gptoss    openai/gpt-oss-20b                        (Apache 2.0, needs a recent transformers)
    gemma     google/gemma-2-9b-it                      (gated: accept the licence, set HF_TOKEN)

Usage:
    python distill_generate.py --teacher deepseek --prompts prompts.jsonl \
        --out out_deepseek.jsonl --limit 20000 --batch 16 [--4bit]

prompts.jsonl: one object per line: {"prompt": str, "lang": "en|hi|te",
"answer": optional verifiable final answer}.

Output line: {"system", "lang", "prompt", "think", "answer", "verified", "teacher"}
(verified = matched a known ground truth; unverified numeric answers go through vote_filter.py)
mapping onto the notebook's chat format
    <|user|>..<|end|><|assistant|><|think|>{think}<|/think|>{answer}<|end|>
Check each teacher's licence before releasing a model trained on its output.
"""
import argparse, json, os, re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

TEACHERS = {
    'deepseek': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-7B',
    'gptoss':   'openai/gpt-oss-20b',
    'gemma':    'google/gemma-2-9b-it',
}
SYSTEM = {
    'en': 'You are VishwamAI, a calm, witty and emotionally aware assistant. '
          'Reason briefly step by step, then give the final answer.',
    'hi': 'आप विश्वमAI हैं, एक शांत और संवेदनशील सहायक। पहले संक्षेप में चरण-दर-चरण सोचें, फिर उत्तर दें।',
    'te': 'మీరు విశ్వమAI, ప్రశాంతమైన సహాయకుడు. ముందు క్లుప్తంగా ఆలోచించి, తర్వాత సమాధానం ఇవ్వండి.',
}


def final_number(s):
    m = re.findall(r'-?\d[\d,]*\.?\d*', s or '')
    return m[-1].replace(',', '').rstrip('.') if m else None


def split_reasoning(teacher, raw):
    """raw is decoded with special tokens kept; returns (think, answer)."""
    if teacher == 'deepseek':                       # R1-distill: <think>...</think>answer
        raw = re.sub(r'<｜[^>]*｜>', '', raw)
        if '</think>' in raw:
            think, ans = raw.split('</think>', 1)
            return think.replace('<think>', '').strip(), ans.strip()
        return '', raw.strip()
    if teacher == 'gptoss':                         # harmony: analysis channel, then final
        m = re.search(r'<\|channel\|>analysis<\|message\|>(.*?)<\|end\|>', raw, re.S)
        f = re.search(r'<\|channel\|>final<\|message\|>(.*?)(?:<\|return\|>|<\|end\|>|$)', raw, re.S)
        return (m.group(1).strip() if m else ''), (f.group(1).strip() if f else '')
    raw = re.sub(r'<[^>]*>', '', raw)               # gemma: no native reasoning trace
    parts = re.split(r'(?i)\bfinal answer\s*:', raw, maxsplit=1)
    return (parts[0].strip(), parts[1].strip()) if len(parts) == 2 else ('', raw.strip())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--teacher', choices=TEACHERS, required=True)
    ap.add_argument('--model', help='override the Hugging Face repo or a local path')
    ap.add_argument('--prompts', required=True)
    ap.add_argument('--out', required=True)
    ap.add_argument('--limit', type=int, default=1000)
    ap.add_argument('--batch', type=int, default=8)
    ap.add_argument('--max-new', type=int, default=1024)
    ap.add_argument('--4bit', dest='four_bit', action='store_true', help='bitsandbytes 4-bit (fits a T4)')
    a = ap.parse_args()

    name = a.model or TEACHERS[a.teacher]
    tok = AutoTokenizer.from_pretrained(name)
    tok.padding_side = 'left'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kw = dict(device_map='auto', torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16)
    if a.four_bit:
        kw['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(name, **kw).eval()

    exs = [json.loads(l) for l in open(a.prompts, encoding='utf-8')][:a.limit]
    seen = set()                                    # resume: skip prompts a previous (interrupted) run already answered
    if os.path.exists(a.out):
        seen = {json.loads(l)['prompt'] for l in open(a.out, encoding='utf-8')}
    exs = [e for e in exs if e['prompt'] not in seen]
    print(f'{len(seen)} already done, {len(exs)} to go', flush=True)
    kept = dropped = 0
    with open(a.out, 'a', encoding='utf-8') as f:
        for i in range(0, len(exs), a.batch):
            chunk = exs[i:i + a.batch]
            texts = []
            for ex in chunk:
                msgs = [{'role': 'user', 'content': SYSTEM.get(ex.get('lang', 'en'), SYSTEM['en'])
                         + '\n\n' + ex['prompt']}]    # system folded into user: Gemma has no system role
                texts.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
            enc = tok(texts, return_tensors='pt', padding=True, add_special_tokens=False).to(model.device)
            with torch.no_grad():
                out = model.generate(**enc, max_new_tokens=a.max_new, do_sample=True,
                                     temperature=0.6, top_p=0.95)
            for ex, ids in zip(chunk, out[:, enc['input_ids'].shape[1]:]):
                think, answer = split_reasoning(a.teacher, tok.decode(ids, skip_special_tokens=False))
                for s in (tok.eos_token, tok.pad_token):
                    answer = answer.replace(s or '', '').strip()
                if not answer:
                    dropped += 1; continue
                # rejection sampling: keep only answers matching a known ground truth
                if ex.get('answer') is not None and final_number(answer) != str(ex['answer']):
                    dropped += 1; continue
                lang = ex.get('lang', 'en')
                f.write(json.dumps(dict(system=SYSTEM.get(lang, SYSTEM['en']), lang=lang,
                                        prompt=ex['prompt'], think=think, answer=answer,
                                        verified=ex.get('answer') is not None,
                                        teacher=a.teacher), ensure_ascii=False) + '\n')
                kept += 1
            f.flush()
            print(f'{i + len(chunk)}/{len(exs)} kept {kept} dropped {dropped}', flush=True)


if __name__ == '__main__':
    main()

In [ ]:
if SKIP_TEACHER:
    print('teacher prompts skipped (SKIP_TEACHER = True)')
else:
    N_PROMPTS = 3000          # start small on a T4; raise on a bigger GPU
    os.makedirs(f'{WORK}/math', exist_ok=True)
    rows = [dict(prompt=q + MATH_SUFFIX, lang='en', answer=gold(a))
            for q, a in zip(gtr['question'][:N_PROMPTS], gtr['answer'][:N_PROMPTS]) if not contaminated(q)]
    with open(f'{WORK}/math/prompts.jsonl', 'w', encoding='utf-8') as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + '\n')
    print(len(rows), 'prompts')


In [ ]:
%%bash -s "$WORK" "$SKIP_TEACHER"
WORK=$1; SKIP=$2
if [ "$SKIP" != "False" ]; then echo "teacher generation skipped (SKIP_TEACHER is not False; value seen: '$SKIP')"; exit 0; fi
if [ ! -f "$WORK/math/prompts.jsonl" ]; then echo "ERROR: $WORK/math/prompts.jsonl is missing - run the prompts cell above first (with SKIP_TEACHER = False)"; exit 1; fi
if [ ! -f distill_generate.py ]; then echo "ERROR: distill_generate.py is missing - run the %%writefile cell above first"; exit 1; fi
PER_GPU=700        # prompts per GPU (raise for more data)
MAXMIN=75          # hard time limit per GPU in minutes; finished batches are kept, re-running resumes
BATCH=48           # lower to 32 if a log shows an out-of-memory error

pkill -f distill_generate.py || true; sleep 2
pip -q install -U bitsandbytes
rm -f $WORK/math/out_deepseek.jsonl
split -n l/2 -d --additional-suffix=.jsonl $WORK/math/prompts.jsonl $WORK/math/part
A=$WORK/math/out_deepseek_a.jsonl; B=$WORK/math/out_deepseek_b.jsonl

CUDA_VISIBLE_DEVICES=0 timeout ${MAXMIN}m python distill_generate.py --teacher deepseek --prompts $WORK/math/part00.jsonl --out $A --limit $PER_GPU --batch $BATCH --4bit > $WORK/math/gen_a.log 2>&1 &
PA=$!
CUDA_VISIBLE_DEVICES=1 timeout ${MAXMIN}m python distill_generate.py --teacher deepseek --prompts $WORK/math/part01.jsonl --out $B --limit $PER_GPU --batch $BATCH --4bit > $WORK/math/gen_b.log 2>&1 &
PB=$!

while kill -0 $PA 2>/dev/null || kill -0 $PB 2>/dev/null; do
  sleep 60
  echo "$(date +%H:%M) | A saved=$(cat $A 2>/dev/null | wc -l) $(tail -n1 $WORK/math/gen_a.log | cut -c1-55) | B saved=$(cat $B 2>/dev/null | wc -l) $(tail -n1 $WORK/math/gen_b.log | cut -c1-55)"
done

wait $PA; SA=$?; wait $PB; SB=$?
echo "FINISHED. exit codes: A=$SA B=$SB   (0 = done, 124 = hit the time limit, other = error, see the logs below)"
wc -l $A $B
tail -n 2 $WORK/math/gen_a.log $WORK/math/gen_b.log
true


## 5 · Build the SFT set (verified teacher solutions + GSM8K human solutions)

In [ ]:
import os, json, re, glob, random
import torch

# --- preflight: stop early with a clear message instead of a NameError ---
_missing = [n for n in ('tok', 'gtr', 'gte', 'encode_prompt', 'make_prompt', 'MATH_SUFFIX', 'WORK', 'contaminated', 'FULL_FT', 'MAX_LEN') if n not in globals()]
if _missing:
    raise RuntimeError("Not defined yet: " + ", ".join(_missing) + ". This session was probably restarted. Run these cells first, in order: "
                       "1) pip install  2) setup (token)  3) GSM8K data and helpers  4) load the student  5) build the SFT set"
                       " - then run this cell again. (You need to run up to: section 3, load the student.)")

def clean_human(a):
    body, ans = a.split('####'); body = re.sub(r'<<.*?>>', '', body).strip()
    return f"{body}\n#### {ans.strip().replace(',', '')}"

pairs = []                                            # (question, target)
for p in sorted(glob.glob(f'{WORK}/math/out_*.jsonl')) + [f'{WORK}/distill/voted.jsonl']:
    if not os.path.exists(p): continue
    for line in open(p, encoding='utf-8'):
        r = json.loads(line)
        if '####' not in r['answer']: continue                    # math rows only
        q = r['prompt'].replace(MATH_SUFFIX, '')
        think = r['think'] if 0 < len(r['think']) <= 2500 else ''
        pairs.append((q, (think + '\n' if think else '') + r['answer']))
n_teacher = len(pairs)
pairs += [(q, clean_human(a)) for q, a in zip(gtr['question'], gtr['answer'])]       # human solutions as extra data
pairs = [(q, t) for q, t in pairs if not contaminated(q)]
random.shuffle(pairs)

examples = []
for q, t in pairs:
    pi = encode_prompt(q)
    ti = tok(' ' + t, add_special_tokens=False)['input_ids'] + [tok.eos_token_id]
    if len(pi) + len(ti) <= MAX_LEN:
        examples.append(dict(input_ids=pi + ti, labels=[-100] * len(pi) + ti))      # loss on the answer only
print(f'{len(examples)} examples ({n_teacher} teacher rows before filtering, {len(gtr)} human GSM8K-train solutions)')
assert len(examples) > 0

def collate(b):
    L = max(len(x['input_ids']) for x in b)
    pad = lambda v, fill: v + [fill] * (L - len(v))
    return dict(input_ids=torch.tensor([pad(x['input_ids'], tok.pad_token_id) for x in b]),
                labels=torch.tensor([pad(x['labels'], -100) for x in b]),
                attention_mask=torch.tensor([[1] * len(x['input_ids']) + [0] * (L - len(x['input_ids'])) for x in b]))

## 6 · LoRA SFT

In [ ]:
# --- preflight: stop early with a clear message instead of a NameError ---
_missing = [n for n in ('FULL_FT', 'model', 'tok', 'examples', 'collate', 'NG', 'WORK', 'SLUG') if n not in globals()]
if _missing:
    raise RuntimeError("Not defined yet: " + ", ".join(_missing) + ". This session was probably restarted. Run these cells first, in order: "
                       "1) pip install  2) setup (token)  3) GSM8K data and helpers  4) load the student  5) build the SFT set"
                       " - then run this cell again. (You need to run up to: section 5, build the SFT set.)")

from transformers import Trainer, TrainingArguments
import inspect, math

if FULL_FT:
    GC = False                                            # 132M model: no need to trade speed for memory
    LR, EPOCHS = 1e-4, 3                                  # 132M from-scratch model: full fine-tune, fp32 weights + fp16 autocast
else:
    from peft import LoraConfig, get_peft_model
    GC = True
    model.gradient_checkpointing_enable(); model.enable_input_require_grads()
    model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type='CAUSAL_LM',
            target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']))
    for p in model.parameters():
        if p.requires_grad: p.data = p.data.float()       # fp16 base + fp32 adapters (required for fp16 training)
    model.print_trainable_parameters()
    LR, EPOCHS = 2e-4, 2

NGPU = max(1, NG)
PER_DEV, ACC = (4, 8) if not FULL_FT else (16, 2)         # tiny model: large per-GPU batch; lower 16 -> 8 if you hit out-of-memory
ACC = max(1, ACC // NGPU)                                  # Trainer uses every visible GPU (DataParallel): keep the effective batch size
EFF = PER_DEV * NGPU * ACC
TOTAL_STEPS = math.ceil(len(examples) / EFF) * EPOCHS
WARMUP = max(1, int(0.03 * TOTAL_STEPS))                   # explicit steps: `warmup_ratio` was removed in newer transformers
print(f'effective batch = {PER_DEV} x {NGPU} GPUs x {ACC} accumulation = {EFF} | {len(examples)} examples | {TOTAL_STEPS} steps | warmup {WARMUP}')

arg_kw = dict(output_dir=f'{WORK}/math/run_{SLUG}', per_device_train_batch_size=PER_DEV,
              gradient_accumulation_steps=ACC, num_train_epochs=EPOCHS, learning_rate=LR,
              lr_scheduler_type='cosine', warmup_steps=WARMUP, fp16=True, logging_steps=20,
              save_strategy='epoch', save_total_limit=1, report_to='none',
              gradient_checkpointing=GC, remove_unused_columns=False,
              group_by_length=True, dataloader_num_workers=2)
supported = set(inspect.signature(TrainingArguments.__init__).parameters)
skipped = [k for k in arg_kw if k not in supported]
if skipped: print('this transformers version does not support, skipping:', skipped)
args = TrainingArguments(**{k: v for k, v in arg_kw.items() if k in supported})

class DS(torch.utils.data.Dataset):
    def __len__(self): return len(examples)
    def __getitem__(self, i): return examples[i]
trainer = Trainer(model=model, args=args, train_dataset=DS(), data_collator=collate)
trainer.train()
OUT_DIR = f'{WORK}/math/{SLUG}-{"full" if FULL_FT else "lora"}'
model.save_pretrained(OUT_DIR); tok.save_pretrained(OUT_DIR)     # full weights for FULL_FT, adapter for LoRA
print('saved', OUT_DIR)


## 7 · Evaluate AFTER and compare

In [ ]:
# --- preflight: stop early with a clear message instead of a NameError ---
_missing = [n for n in ('model', 'evaluate', 'base_acc', 'examples', 'STUDENT', 'N_EVAL', 'FULL_FT', 'WORK') if n not in globals()]
if _missing:
    raise RuntimeError("Not defined yet: " + ", ".join(_missing) + ". This session was probably restarted. Run these cells first, in order: "
                       "1) pip install  2) setup (token)  3) GSM8K data and helpers  4) load the student  5) build the SFT set"
                       " - then run this cell again. (You need to run up to: section 6, training (and run the baseline cell in section 3 or set base_acc = 0.013).)")

model.config.use_cache = True
after_acc = evaluate(model)
print(f"{'':28s}{'GSM8K (greedy, ' + str(N_EVAL) + ' q)':>26s}")
print(f"{'student before':28s}{100 * base_acc:>25.1f}%")
print(f"{'after distillation + SFT':28s}{100 * after_acc:>25.1f}%")
print(f"{'change':28s}{100 * (after_acc - base_acc):>+25.1f} points   (noise is about +-3 points at n={N_EVAL})")
print('target: 50%  ->', 'reached' if after_acc >= 0.5 else 'not reached')
rp = f'{WORK}/math/results.json'
R = json.load(open(rp)) if os.path.exists(rp) else {}
R[STUDENT] = dict(before=base_acc, after=after_acc, n_eval=N_EVAL, full_ft=FULL_FT, examples=len(examples))
json.dump(R, open(rp, 'w'), indent=1)
print('\nall runs so far (GSM8K greedy):')
for k, v in R.items():
    print(f"  {k:36s} before {100 * v['before']:5.1f}%  after {100 * v['after']:5.1f}%  (n={v['n_eval']}, {v['examples']} examples)")


## 8 · Publish (optional, private by default)
Set `LICENSE` yourself. The model is derived from `STUDENT` and trained on outputs of the teachers, so its licence must be compatible with theirs.

In [ ]:
from huggingface_hub import HfApi

# Choose the licence yourself. 'other' is a neutral placeholder for a PRIVATE experiment; decide properly before making it public
# (the base model's pre-training data and GSM8K (MIT) each have their own terms, and any teacher model used has its own licence).
LICENSE = 'other'
PRIVATE = True
REPO_NAME = f'{SLUG}-gsm8k-sft-v0'

assert os.environ.get('HF_TOKEN'), 'no Hugging Face token in this session: run the setup cell / paste a token with WRITE access'
api = HfApi(token=os.environ['HF_TOKEN'])
repo = f"{api.whoami()['name']}/{REPO_NAME}"

n_t = globals().get('n_teacher', 0)                       # teacher rows that went into training (0 if generation was skipped)
data_line = ("human-written GSM8K train solutions only (no teacher-model data was used)" if not n_t else
             f"human-written GSM8K train solutions plus {n_t} solutions from DeepSeek-R1-Distill-Qwen-7B filtered for correct final answers")
kind = ('Full fine-tune of the from-scratch VishwamAI base' if FULL_FT else f'LoRA adapter for {STUDENT}')
card = f"""---
license: {LICENSE}
base_model: {STUDENT}
tags: [math, gsm8k, experimental]
---
# {REPO_NAME}

{kind} (`{STUDENT}`), trained on {data_line}: {len(examples)} examples. GSM8K training questions were checked for
8-gram overlap with the GSM8K test set and overlapping ones were removed.

GSM8K greedy accuracy on {N_EVAL} test questions: {100 * base_acc:.1f}% before, {100 * after_acc:.1f}% after
(about +-3 points of noise at this sample size). This is an experimental small model with very limited math ability.
"""
open(f'{OUT_DIR}/README.md', 'w', encoding='utf-8').write(card)
print(card)
api.create_repo(repo, private=PRIVATE, exist_ok=True)
api.upload_folder(folder_path=OUT_DIR, repo_id=repo, commit_message='Upload fine-tuned model')
print('https://huggingface.co/' + repo, '(private)' if PRIVATE else '(public)')
